# Emotion Detection Pipeline Interface

Run this notebook to test the audio emotion detection system.

In [1]:
import asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output
from orchestrator import EmotionOrchestrator
import os

# Initialize Orchestrator
orchestrator = EmotionOrchestrator()

print("System Initialized.")

System Initialized.


In [ ]:
# UI Components
current_file_display = widgets.Text(
    value='Ready to start',
    description='Current File:',
    disabled=True,
    layout=widgets.Layout(width='100%')
)

status_display = widgets.HTML(
    value="<b>Status:</b> <span style='color: gray;'>System Ready</span>",
    layout=widgets.Layout(margin='10px 0')
)

results_display = widgets.HTML(
    value="<i>Results will appear here...</i>",
    layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='10px 0', min_height='150px')
)

run_btn = widgets.Button(
    description='Start / Next File',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='100%', height='40px')
)

output_area = widgets.Output(layout={'border': '1px solid #eee', 'margin': '10px 0'})

# Feedback Buttons
btn_layout = widgets.Layout(width='100%')
btn_openai = widgets.Button(description='OpenAI Better', button_style='success', layout=btn_layout)
btn_local = widgets.Button(description='Local Better', button_style='info', layout=btn_layout)
btn_same = widgets.Button(description='Same / Equal', button_style='warning', layout=btn_layout)
btn_bad = widgets.Button(description='Both Bad', button_style='danger', layout=btn_layout)

feedback_box = widgets.GridBox(
    children=[btn_openai, btn_local, btn_same, btn_bad],
    layout=widgets.Layout(
        grid_template_columns='1fr 1fr 1fr 1fr',
        grid_gap='10px',
        visibility='hidden',
        margin='10px 0'
    )
)

current_result = None

def update_status(msg, color="blue"):
    status_display.value = f"<b>Status:</b> <span style='color: {color};'>{msg}</span>"

def show_results(result):
    global current_result
    current_result = result
    
    # Update display with filename
    if "file" in result:
        current_file_display.value = os.path.basename(result['file'])
    
    if "error" in result:
        update_status(f"Error: {result['error']}", "red")
        results_display.value = f"<div style='color: red;'><b>Error:</b> {result['error']}</div>"
        return

    # Construct HTML for results
    openai = result.get('openai', {})
    local_results = result.get('local', [])
    top_local = local_results[0] if local_results else {'label': 'N/A', 'score': 0.0}

    html = f"""
    <div style='display: flex; gap: 20px;'>
        <div style='flex: 1; border: 1px solid #4CAF50; padding: 10px; border-radius: 5px;'>
            <h3 style='margin-top: 0; color: #4CAF50;'>OpenAI Analysis</h3>
            <p><b>Emotion:</b> {openai.get('emotion', 'N/A')}</p>
            <p><b>Confidence:</b> {openai.get('confidence', 'N/A')}</p>
        </div>
        <div style='flex: 1; border: 1px solid #2196F3; padding: 10px; border-radius: 5px;'>
            <h3 style='margin-top: 0; color: #2196F3;'>Local Model</h3>
            <p><b>Emotion:</b> {top_local.get('label', 'N/A')}</p>
            <p><b>Score:</b> {top_local.get('score', 0):.4f}</p>
        </div>
    </div>
    <p style='margin-top: 15px; text-align: center; color: #666;'><i>Please vote on the quality below:</i></p>
    """
    results_display.value = html
    update_status("Ready for feedback", "green")
    feedback_box.layout.visibility = 'visible'

async def on_run_click(b):
    results_display.value = "<i>Processing...</i>"
    feedback_box.layout.visibility = 'hidden'
    
    # Note: Backend logs will still appear in output_area due to orchestrator prints
    with output_area:
        clear_output()
        try:
            # run_analysis(None) triggers auto-selection
            res = await orchestrator.run_analysis(status_callback=lambda m: update_status(m))
            show_results(res)
        except Exception as e:
            update_status(f"Processing Error: {e}", "red")
            results_display.value = f"<div style='color: red;'>Processing Error: {e}</div>"

def on_feedback_click(pref):
    if current_result:
        orchestrator.save_feedback(current_result, pref)
        results_display.value = f"<div style='text-align: center; color: #666; padding: 20px;'>Feedback <b>{pref}</b> saved!<br>Click 'Start / Next File' to continue.</div>"
        update_status("Feedback saved.", "gray")
        feedback_box.layout.visibility = 'hidden'

run_btn.on_click(lambda b: asyncio.create_task(on_run_click(b)))

btn_openai.on_click(lambda b: on_feedback_click("openai"))
btn_local.on_click(lambda b: on_feedback_click("local"))
btn_same.on_click(lambda b: on_feedback_click("same"))
btn_bad.on_click(lambda b: on_feedback_click("bad"))

ui = widgets.VBox([
    widgets.HTML("<h2>Emotion Detection Dashboard</h2>"),
    current_file_display, 
    status_display, 
    run_btn, 
    results_display, 
    feedback_box,
    widgets.HTML("<hr><b>Backend Logs:</b>"),
    output_area
], layout=widgets.Layout(width='800px', margin='0 auto'))

display(ui)